# Allelic Dosage Effect Analysis Dashboard

This notebook performs trait-specific allelic dosage analysis by integrating GWAS results, genotype (taglo) data, and phenotype measurements.

For each selected trait, significant tag loci are extracted from GWAS results and used to compute per-variety allelic dosage. These dosages are merged with phenotype values to evaluate how trait expression changes with increasing allele count.

The workflow includes:

Visualization of trait values across dosage groups (boxplots)
Detection of outliers within dosage groups (IQR-based)
Filtering of dosage groups based on consistent trend (increasing/decreasing effect)
Calculation of effect sizes relative to baseline dosage (0)
Computation of fold change between baseline and strongest effect

An interactive interface allows users to explore traits dynamically, inspect dosage-response patterns, and assess the strength and consistency of allelic effects.

In [0]:
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from IPython.display import display, clear_output
import ipywidgets as widgets

In [0]:


# ============================================================
# 0) LOAD CONFIG (ONLY SOURCE OF TRUTH)
# ============================================================



CONFIG_PATH = "/Volumes/bmqg/default_bronze/fatemeh/config_mixed.yaml"

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)


GWAS_TABLE = CONFIG["data"]["gwas_table_newharvested"]

TAGLO_TABLE =CONFIG["paths"]["TAGLO_TABLE"]
PHENO_PATH = CONFIG["paths"]["aroma_matrix_newharvested"]
df_pheno = pd.read_csv(PHENO_PATH)


# ============================================================
# 0.5) LOAD PHENOTYPES ONCE (avoid reading CSV every interaction)
# ============================================================

PHENO_ALL = pd.read_csv(PHENO_PATH)
PHENO_ALL["Variety"] = PHENO_ALL["Variety"].astype(str).str.strip()


# ============================================================
# 1) CORE PLOT FUNCTION (allelic dosage)
# ============================================================

def plot_taglo_effect(spark, trait, taglo_ids, pheno_all, taglo_table):
    # ---------- phenotypes ----------
    if trait not in pheno_all.columns:
        raise KeyError(f"Trait '{trait}' not found in phenotype columns.")

    ph = pheno_all[["Variety", trait]].dropna().copy()
    ph["Variety"] = ph["Variety"].astype(str).str.lower().str.strip()
    ph = ph.rename(columns={trait: "trait_value"})

    # ---------- genotypes (robust taglo_id typing + normalize variety) ----------
    ids_long = [int(x) for x in taglo_ids]
    ids_str  = [str(x) for x in taglo_ids]

    base = spark.table(taglo_table)

    gt_spark = (
        base
        .filter(
            (F.col("taglo_id").cast("long").isin(ids_long)) |
            (F.col("taglo_id").cast("string").isin(ids_str))
        )
        .select(
            F.lower(F.trim(F.col("variety").cast("string"))).alias("Variety"),
            F.col("value").cast("double").alias("value"),
        )
        .groupBy("Variety")
        .agg(F.round(F.avg("value")).cast("int").alias("dosage"))
    )

    gt = gt_spark.toPandas()

    # ---------- merge ----------
    df = ph.merge(gt, on="Variety", how="left")

    df_plot = df.dropna(subset=["dosage"]).copy()
    df_plot["dosage"] = df_plot["dosage"].astype(int)

    if df_plot.empty:
        print("No dosage matched after merge. Check taglo_id match + Variety naming.")
        print("Phenotype varieties sample:", ph["Variety"].head(5).tolist())
        print("Genotype varieties sample:", gt["Variety"].head(5).tolist() if not gt.empty else "gt empty")
        return df, df_plot

    # ---------- plot ----------
    plt.figure(figsize=(6, 4))
    df_plot.boxplot(column="trait_value", by="dosage")
    plt.title(trait)
    plt.suptitle("")
    plt.xlabel("Allelic dosage")
    plt.ylabel("trait_value")
    plt.tight_layout()
    plt.show()

    return df, df_plot





# ============================================================
# 2) TREND-CONSISTENT FILTER
# ============================================================

def trend_consistent_allelic_filter(df, baseline=0, min_group_size=3, stat="median"):
    agg = (
        df.groupby("dosage")["trait_value"]
        .agg(count="count", median="median", mean="mean")
        .reset_index()
        .sort_values("dosage")
    )

    ok = agg[agg["count"] >= min_group_size]
    if baseline not in ok["dosage"].values:
        return df[df["dosage"] == baseline], agg, {
            "status": "insufficient_support",
            "kept_dosages": [baseline],
        }

    vals = ok.set_index("dosage")[stat]
    diffs = vals.diff().dropna()

    if diffs.empty:
        return df, agg, {"status": "flat"}

    direction = np.sign(diffs.median())
    keep = [baseline]
    last = vals.loc[baseline]

    for d in vals.index:
        if d == baseline:
            continue
        if np.sign(vals.loc[d] - last) == direction:
            keep.append(d)
            last = vals.loc[d]

    return (
        df[df["dosage"].isin(keep)],
        agg,
        {"status": "ok", "kept_dosages": keep}
    )


# ============================================================
# 3) EFFECT SIZE + FOLD CHANGE
# ============================================================

def compute_effect_and_fc(df, baseline=0, stat="median"):
    g = df.groupby("dosage")["trait_value"]
    if baseline not in g.groups:
        return {}, np.nan

    base = getattr(g.get_group(baseline), stat)()

    effects = {int(d): getattr(v, stat)() - base for d, v in g}
    max_d = max(effects, key=lambda k: abs(effects[k])) if effects else baseline

    base_stat = getattr(g.get_group(baseline), stat)()
    max_stat  = getattr(g.get_group(max_d), stat)()

    fc = (max_stat / base_stat) if base_stat not in [0, np.nan] else np.nan
    return effects, fc


# ============================================================
# 4) BUILD TRAIT → TAGLO IDS TABLE (FROM GWAS, CONFIG-DRIVEN)
# ============================================================


P_COL = "p_wald"   
P_TH  = 1e-6       # threshold

gwas_df = spark.table(GWAS_TABLE)

if P_COL not in gwas_df.columns:
    raise KeyError(f"GWAS table does not contain column '{P_COL}'. Available columns: {gwas_df.columns}")

res = (
    gwas_df
    .filter(F.col(P_COL) < F.lit(P_TH))
    .select(
        "trait",
        F.array("taglo_id1", "taglo_id2", "taglo_id3", "taglo_id4").alias("taglo_ids")
    )
    .withColumn("taglo_ids", F.expr("filter(taglo_ids, x -> x is not null and x > 0)"))
    .groupBy("trait")
    # collect ALL taglo_ids across rows, then flatten, then unique
    .agg(F.array_distinct(F.flatten(F.collect_list("taglo_ids"))).alias("taglo_ids"))
    .filter(F.size("taglo_ids") > 0)
    .toPandas()
)

print("Traits ready:", len(res))


# ============================================================
# 4.5) OUTLIER DETECTION (IQR within dosage groups)
# ============================================================

def find_outliers_iqr_by_group(df, group_col="dosage", value_col="trait_value", k=1.5, min_group_size=4):
    out_rows = []
    for d, g in df.groupby(group_col):
        g = g.dropna(subset=[value_col])
        if len(g) < min_group_size:
            continue
        q1 = g[value_col].quantile(0.25)
        q3 = g[value_col].quantile(0.75)
        iqr = q3 - q1
        if iqr == 0 or np.isnan(iqr):
            continue
        lo = q1 - k * iqr
        hi = q3 + k * iqr
        o = g[(g[value_col] < lo) | (g[value_col] > hi)].copy()
        if not o.empty:
            o["outlier_rule"] = f"IQR(k={k})"
            o["lo"] = lo
            o["hi"] = hi
            out_rows.append(o)
    if out_rows:
        return pd.concat(out_rows, ignore_index=True)
    return df.iloc[0:0].copy()


# ============================================================
# 5) DASHBOARD
# ============================================================

def run_trait_dashboard(trait, min_group_size=3):
    clear_output(wait=True)

    row = res[res["trait"] == trait].iloc[0]
    taglo_ids = row["taglo_ids"]

    print(f"Trait: {trait}")
    print(f"Number of taglo_ids: {len(taglo_ids)}")

    df_raw, df = plot_taglo_effect(
        spark=spark,
        trait=trait,
        taglo_ids=taglo_ids,
        pheno_all=PHENO_ALL,
        taglo_table=TAGLO_TABLE
    )

    if df.empty:
        print("No merged data after dropping missing dosages. (No genotype rows matched?)")
        return

    # ---- OUTLIERS ----
    out_df = find_outliers_iqr_by_group(df, k=1.5, min_group_size=max(4, min_group_size))

    if out_df.empty:
        print("\nOutliers: none detected (IQR rule).")
    else:
        print("\nOutlier Varieties (IQR within dosage):")
        display(
            out_df[["Variety", "dosage", "trait_value", "lo", "hi", "outlier_rule"]]
            .sort_values(["dosage", "trait_value"])
            .reset_index(drop=True)
        )

    df_filt, summary, report = trend_consistent_allelic_filter(
        df,
        baseline=0,
        min_group_size=min_group_size
    )

    effect, fc = compute_effect_and_fc(df_filt)

    print("\nREPORT:", report)
    print("Effect:", effect)
    print("Fold change:", fc)
    display(summary)


# ============================================================
# 6) INTERACTIVE UI
# ============================================================

trait_dd = widgets.Dropdown(
    options=sorted(res["trait"].astype(str).tolist()),
    description="Trait:"
)

min_n_slider = widgets.IntSlider(
    value=3, min=2, max=10, step=1,
    description="min n"
)

widgets.interact(
    run_trait_dashboard,
    trait=trait_dd,
    min_group_size=min_n_slider
);


Traits ready: 32


interactive(children=(Dropdown(description='Trait:', options=('(E)-2-Decenal', '1-Hexanol, 2-ethyl-', '1-Octen…